# 1.0 Exploración de viajes MiBici (2022-01 a 2026-09)

Notebook solo de exploración. La lógica final vive en el módulo `mibici` (ingesta en `dataset.py`, variables en `features.py`).

Requisito: haber corrido `python -m mibici.dataset` y `python -m mibici.features`.

In [ ]:
import json

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from mibici.config import DATA_MANIFEST, PANEL_PARQUET, STATIONS_PARQUET, TRIPS_INTERIM_DIR

TRIPS = str(TRIPS_INTERIM_DIR / 'trips_*.parquet')
manifest = json.loads(DATA_MANIFEST.read_text())
print('Versión de datos:', manifest['data_version'])
manifest['totals']

## Calidad de la ingesta por mes
Codificación, formato de fecha y registros eliminados en la limpieza.

In [ ]:
files = pd.DataFrame(manifest['files'])
files[['month', 'encoding', 'date_format', 'rows_raw', 'rows_clean', 'dropped_duration', 'dropped_failed_dock', 'dropped_unknown_station']]

## Viajes por mes

In [ ]:
monthly = duckdb.sql(f"SELECT date_trunc('month', inicio) AS mes, count(*) AS viajes FROM read_parquet('{TRIPS}') GROUP BY 1 ORDER BY 1").df()
ax = monthly.plot(x='mes', y='viajes', figsize=(10, 3), legend=False, title='Viajes por mes')
ax.set_ylabel('viajes');

## Perfil horario: día hábil vs. fin de semana
Motiva las variables de calendario de P1 y los perfiles de P3.

In [ ]:
hourly = duckdb.sql(f"""SELECT hour(inicio) AS hora, dayofweek(inicio) IN (0, 6) AS fin_de_semana, count(*) AS viajes
FROM read_parquet('{TRIPS}') GROUP BY ALL""").df()
hourly.pivot(index='hora', columns='fin_de_semana', values='viajes').rename(columns={False: 'hábil', True: 'fin de semana'}).plot(figsize=(8, 3), title='Viajes por hora de inicio');

## Desbalance por estación (P2)
Distribución del flujo neto horario relativo a la capacidad y prevalencia de la etiqueta proxy.

In [ ]:
duckdb.sql(f"SELECT p2_label, count(*) AS horas, round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct FROM read_parquet('{PANEL_PARQUET}') GROUP BY 1").df()

## Estaciones en el mapa

In [ ]:
st = pd.read_parquet(STATIONS_PARQUET)
fig, ax = plt.subplots(figsize=(6, 6))
for m, g in st.groupby('municipio'):
    ax.scatter(g['lon'], g['lat'], s=8, label=m)
ax.legend(); ax.set_title('Estaciones por municipio');

## Distribución de duración de viajes
Densidad y CDF. Revela el pico típico de ~10 min en ciclovías urbanas.

In [ ]:
import matplotlib.ticker as mticker
import numpy as np

dur_df = duckdb.sql(
    f"SELECT (duracion_s / 60.0) AS duracion_min "
    f"FROM read_parquet('{TRIPS}') USING SAMPLE 200000"
).df()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(dur_df['duracion_min'], bins=120,
             range=(0, 60), color='steelblue', edgecolor='none')
axes[0].set_xlabel('Duración (min)')
axes[0].set_ylabel('Frecuencia (muestra 200 k)')
axes[0].set_title('Distribución de duración (0-60 min)')
med = dur_df['duracion_min'].median()
axes[0].axvline(med, color='tomato', linestyle='--',
                label=f'Mediana: {med:.1f} min')
axes[0].legend()

axes[1].hist(dur_df['duracion_min'], bins=80,
             range=(0, 180), color='steelblue', edgecolor='none',
             cumulative=True, density=True)
axes[1].set_xlabel('Duración (min)')
axes[1].set_ylabel('Proporción acumulada')
axes[1].set_title('CDF de duración de viajes')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1))

fig.tight_layout()


## Top 20 pares Origen-Destino más frecuentes
Pares con mayor flujo identifican los corredores clave de la red.

In [ ]:
od = duckdb.sql(
    f"SELECT origen_id, destino_id, count(*) AS viajes "
    f"FROM read_parquet('{TRIPS}') "
    f"WHERE origen_id != destino_id "
    f"GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 20"
).df()
od['par'] = od['origen_id'].astype(str) + ' > ' + od['destino_id'].astype(str)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(od['par'][::-1], od['viajes'][::-1], color='steelblue')
ax.set_xlabel('Viajes totales')
ax.set_title('Top 20 pares Origen > Destino')
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
)
fig.tight_layout()


## Viajes por día de la semana
Cuantifica la caída del fin de semana y motiva el feature `dow`.

In [ ]:
dow_df = duckdb.sql(
    f"SELECT dayofweek(inicio) AS dow, count(*) AS viajes "
    f"FROM read_parquet('{TRIPS}') GROUP BY 1 ORDER BY 1"
).df()

DIAS = ['Dom', 'Lun', 'Mar', 'Mie', 'Jue', 'Vie', 'Sab']
dow_df['dia'] = dow_df['dow'].map(lambda d: DIAS[d])

colors_dow = ['#d62728' if d in [0, 6] else '#1f77b4'
              for d in dow_df['dow']]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(dow_df['dia'], dow_df['viajes'] / 1e6, color=colors_dow)
ax.set_ylabel('Viajes (millones)')
ax.set_title('Viajes totales por dia de la semana')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#1f77b4', label='Dia habil'),
    Patch(color='#d62728', label='Fin de semana'),
])
fig.tight_layout()


## Capacidad por municipio
Capacidad total y número de estaciones por municipio.

In [ ]:
st = pd.read_parquet(STATIONS_PARQUET)
cap = st.groupby('municipio')['capacity'].agg(
    total='sum', estaciones='count', media='mean'
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cap['total'].sort_values().plot.barh(ax=axes[0], color='teal')
axes[0].set_xlabel('Capacidad total (anclajes)')
axes[0].set_title('Capacidad total por municipio')

cap['estaciones'].sort_values().plot.barh(ax=axes[1], color='darkorange')
axes[1].set_xlabel('Numero de estaciones')
axes[1].set_title('Estaciones por municipio')
fig.tight_layout()


## Distribución de la clase P2 por hora del día
Los picos de la mañana y la tarde concentran más eventos de riesgo.

In [ ]:
p2_hour = duckdb.sql(
    f"SELECT hour(ts) AS hora, p2_label, count(*) AS horas "
    f"FROM read_parquet('{PANEL_PARQUET}') GROUP BY 1, 2 ORDER BY 1"
).df()

p2_pivot = p2_hour.pivot(index='hora', columns='p2_label',
                         values='horas').fillna(0)
p2_prop = p2_pivot.div(p2_pivot.sum(axis=1), axis=0)

COLOR_MAP = {
    'normal': '#4c72b0',
    'vaciado': '#dd8452',
    'saturacion': '#55a868',
}
fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(p2_prop))
for lbl in ['normal', 'vaciado', 'saturacion']:
    if lbl in p2_prop.columns:
        ax.bar(p2_prop.index, p2_prop[lbl], bottom=bottom,
               label=lbl, color=COLOR_MAP[lbl])
        bottom += p2_prop[lbl].values
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
ax.set_xlabel('Hora del dia')
ax.set_ylabel('Proporcion de estacion-horas')
ax.set_title('Composicion de etiqueta P2 por hora del dia')
ax.legend(loc='upper right')
ax.set_xticks(range(24))
fig.tight_layout()


## Correlación de features del panel
Detecta colinealidad antes de modelar (muestra 100 k).

In [ ]:
import seaborn as sns

panel_sample = duckdb.sql(
    f"SELECT departures, arrivals, net_flow, capacity, "
    f"dep_lag_24h, dep_lag_168h, arr_lag_24h, arr_lag_168h, "
    f"dep_mean_7d, dep_mean_28d, netrel_mean_28d "
    f"FROM read_parquet('{PANEL_PARQUET}') USING SAMPLE 100000"
).df()

corr = panel_sample.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Correlacion entre features numericas')
fig.tight_layout()


## Top 15 estaciones por flujo neto medio
Mayor |flujo neto| => más candidatas a rebalanceo frecuente.

In [ ]:
net_st = duckdb.sql(
    f"SELECT station_id, avg(net_flow) AS net_flow_medio "
    f"FROM read_parquet('{PANEL_PARQUET}') "
    f"GROUP BY 1 ORDER BY abs(avg(net_flow)) DESC LIMIT 15"
).df()

colors_net = ['#d62728' if v < 0 else '#2ca02c'
              for v in net_st['net_flow_medio']]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(net_st['station_id'].astype(str)[::-1],
        net_st['net_flow_medio'][::-1],
        color=colors_net[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Flujo neto medio por hora')
ax.set_title('Top 15 estaciones por |flujo neto medio|')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#2ca02c', label='Neto positivo (mas llegadas)'),
    Patch(color='#d62728', label='Neto negativo (mas salidas)'),
])
fig.tight_layout()


## Estacionalidad mensual del desbalance (P2)
¿En qué meses del año ocurre más riesgo de vaciado o saturación?

In [ ]:
monthly_risk = duckdb.sql(
    f"SELECT month(ts) AS mes, p2_label, count(*) AS n "
    f"FROM read_parquet('{PANEL_PARQUET}') GROUP BY 1, 2 ORDER BY 1"
).df()

risk_pivot = monthly_risk.pivot(index='mes', columns='p2_label',
                                values='n').fillna(0)
risk_prop = risk_pivot.div(risk_pivot.sum(axis=1), axis=0)

MESES = ['Ene','Feb','Mar','Abr','May','Jun',
         'Jul','Ago','Sep','Oct','Nov','Dic']
fig, ax = plt.subplots(figsize=(10, 4))
bottom2 = np.zeros(len(risk_prop))
for lbl in ['normal', 'vaciado', 'saturacion']:
    if lbl in risk_prop.columns:
        ax.bar(risk_prop.index, risk_prop[lbl], bottom=bottom2,
               label=lbl, color=COLOR_MAP[lbl])
        bottom2 += risk_prop[lbl].values
ax.set_xticks(risk_prop.index)
ax.set_xticklabels([MESES[m - 1] for m in risk_prop.index])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
ax.set_xlabel('Mes')
ax.set_title('Composicion de etiqueta P2 por mes del anio')
ax.legend(loc='upper right')
fig.tight_layout()
